# Unsloth QLoRA Fine-tuning

Google Colab에서 GPU 런타임을 선택한 후 순서대로 실행합니다. 실제 학습 전 `data/final/train.jsonl`을 업로드해야 합니다.

In [ ]:
!pip install -q unsloth datasets trl

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

In [ ]:
dataset = load_dataset("json", data_files={"train": "data/final/train.jsonl", "validation": "data/final/validation.jsonl"})

def format_messages(examples):
    return {"text": [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False) for messages in examples["messages"]]}

dataset = dataset.map(format_messages, batched=True)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=25,
        save_steps=25,
        output_dir="outputs",
        report_to="none",
        seed=3407,
    ),
)
trainer.train()

In [ ]:
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")
# 학습 완료 후 필요하면 GGUF export:
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")